# 🎭 02 — Fleet Dance (Choreography)

Send timed, coordinated commands to the entire online fleet.  
Uses `broadcast()` for fan-out and `sequence()` for choreographed step-delay routines.

> **Note:** Bots that are offline are automatically skipped — no error handling needed.

---

## Setup

In [ ]:
import sys, time
from pathlib import Path

REPO_ROOT = Path().resolve().parent
FLEET_MGR = REPO_ROOT / 'fleet-manager'
if str(FLEET_MGR) not in sys.path:
    sys.path.insert(0, str(FLEET_MGR))

from fleet_manager import scan, broadcast, sequence, send_command
from fleet_manager.discovery import Fleet, Bot

FLEET_YAML = REPO_ROOT / 'docs' / 'fleet.yaml'
print('fleet_manager imported ✓')

---
## 1 · Discover Online Fleet

In [ ]:
BOT_FAMILIES = ['rfbot', 'mybot', 'carbot', 'paulbot', 'simplebot', 'dogbot']

t0 = time.perf_counter()
fleet = scan(BOT_FAMILIES, workers=20, timeout=1.0, yaml_path=FLEET_YAML)
elapsed = time.perf_counter() - t0

print(fleet.summary())
print(f'Scan completed in {elapsed:.2f}s')

if not fleet.online:
    print('\n⚠️  No bots online. Choreography cells will run but have no targets.')
else:
    for bot in fleet.online:
        print(f'  ● {bot.hostname:<22}  {bot.ip}')

---
## 2 · Single Broadcast — LED Flash

Send one command to all online bots simultaneously.

In [ ]:
# POST /api/led/color → { "r": 0, "g": 255, "b": 0 }
# Bots that are offline are silently skipped.

results = broadcast(
    fleet,
    endpoint='led/color',
    payload={'r': 0, 'g': 255, 'b': 0},   # green
    timeout=3.0,
)

for hostname, resp in results.items():
    status = 'OK' if resp is not None else 'no response'
    print(f'  {hostname:<22}  {status}')

if not results:
    print('  (no online bots to command)')

---
## 3 · Sequence — Choreographed Routine

A step-by-step routine with delays between each command.

Format: `(endpoint, payload_dict, delay_after_seconds)`

In [ ]:
DANCE_ROUTINE = [
    # ── Attention flash ──────────────────────────────────────────────────────
    ('led/color',  {'r': 255, 'g': 255, 'b': 255}, 0.3),   # white flash
    ('led/color',  {'r': 0,   'g': 0,   'b': 0},   0.3),   # off
    ('led/color',  {'r': 255, 'g': 255, 'b': 255}, 0.3),   # white flash
    ('led/color',  {'r': 0,   'g': 0,   'b': 0},   0.5),   # off

    # ── Red — ready ──────────────────────────────────────────────────────────
    ('led/color',  {'r': 255, 'g': 0,   'b': 0},   1.0),   # red

    # ── Wave servo left ──────────────────────────────────────────────────────
    ('servo/move', {'joint': 'head', 'angle': -30, 'speed': 50}, 0.8),
    ('servo/move', {'joint': 'head', 'angle':  30, 'speed': 50}, 0.8),
    ('servo/move', {'joint': 'head', 'angle':   0, 'speed': 50}, 0.4),

    # ── Green — done ─────────────────────────────────────────────────────────
    ('led/color',  {'r': 0,   'g': 255, 'b': 0},   1.0),   # green

    # ── Home position ────────────────────────────────────────────────────────
    ('servo/home', None,                             0.5),
    ('led/color',  {'r': 0,   'g': 0,   'b': 0},   0.0),   # off
]

print(f'Running {len(DANCE_ROUTINE)}-step dance routine on {len(fleet.online)} bot(s)…')

t0 = time.perf_counter()
sequence(fleet, DANCE_ROUTINE, timeout=3.0)
elapsed = time.perf_counter() - t0

print(f'Routine complete in {elapsed:.2f}s')

---
## 4 · Per-Bot Command (Single Target)

In [ ]:
# Pick the first online bot (if any)
target = fleet.online[0] if fleet.online else None

if target:
    print(f'Sending custom command to {target.hostname} ({target.ip})…')
    resp = send_command(
        target,
        endpoint='servo/move',
        payload={'joint': 'tail', 'angle': 45, 'speed': 80},
    )
    print(f'Response: {resp}')
else:
    print('No online bots to target.')

---
## 5 · Timing Diagram

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    steps      = [s[0].replace('/', '\n') for s in DANCE_ROUTINE]
    delays     = [s[2] for s in DANCE_ROUTINE]
    cumulative = list(np.cumsum([0] + delays[:-1]))

    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor('#0f172a')
    ax.set_facecolor('#1e293b')

    for i, (start, dur, label) in enumerate(zip(cumulative, delays, steps)):
        color = '#4ade80' if 'led' in label else '#60a5fa' if 'servo' in label else '#f472b6'
        ax.barh(0, dur, left=start, height=0.6, color=color, edgecolor='#0f172a', linewidth=1.5)
        if dur > 0.15:
            ax.text(start + dur / 2, 0, label,
                    ha='center', va='center', fontsize=7.5,
                    color='#0f172a', fontweight='bold')

    ax.set_yticks([])
    ax.set_xlabel('Time (seconds)', color='#94a3b8')
    ax.set_title('Dance Routine — Timing Diagram', color='white', fontsize=13)
    ax.tick_params(colors='#94a3b8')
    for sp in ax.spines.values():
        sp.set_color('#334155')

    patches = [
        mpatches.Patch(color='#4ade80', label='LED'),
        mpatches.Patch(color='#60a5fa', label='Servo'),
        mpatches.Patch(color='#f472b6', label='Other'),
    ]
    ax.legend(handles=patches, loc='lower right',
              facecolor='#1e293b', edgecolor='#334155', labelcolor='#94a3b8')

    plt.tight_layout()
    plt.show()

except ImportError:
    print('matplotlib not installed — run: pip install matplotlib')